# Two-well energy-based model

Show that evaluating an energy is easy while obtaining representative samples depends on Langevin step size, noise, burn-in, and mixing.

**Classroom pattern:** predict first, run one cell, inspect the intermediate state, then change one parameter. All random seeds are fixed for the first run.

## Before you run

- Use the environment from `README.md`; no lecturer-specific Conda environment is required.
- Set `LECTURE2_QUICK=1` before starting Jupyter for the CPU-safe verification path.
- CUDA is selected first, then Apple MPS, then CPU. A separate MLX installation is not required.
- Read the prediction prompt before each experiment. The fixed-seed result is a mechanism check, not a quality benchmark.
- Every generated file stays inside `fallbacks/` or `checkpoints/`. No cell uploads or publishes anything.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "demo_models.py").exists():
    candidate = ROOT / "lecture 2" / "demos"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))
(ROOT / "fallbacks").mkdir(exist_ok=True)
print(f"Demo root: {ROOT}")

import numpy as np
import matplotlib.pyplot as plt

## 1. Energy, gradient, and the two target modes

In [ ]:
def energy(x):
    return (x**2 - 1.0)**2

def grad_energy(x):
    return 4.0 * x * (x**2 - 1.0)

grid = np.linspace(-2.2, 2.2, 700)
plt.figure(figsize=(8, 3.2))
plt.plot(grid, energy(grid), lw=2)
plt.scatter([-1, 1], [0, 0], s=70, label="low-energy modes")
plt.scatter([0], [1], s=70, label="barrier / zero gradient")
plt.ylim(-0.1, 4); plt.xlabel("x"); plt.ylabel("E(x)")
plt.legend(); plt.title("E(x) = (x² − 1)²")
plt.tight_layout(); plt.show()
print("∇E(0) =", grad_energy(np.array([0.0]))[0])

## 2. One Langevin transition

Drift moves downhill in energy. Noise prevents the procedure from becoming pure optimisation.

In [ ]:
def langevin(initial, steps=400, eta=0.02, noise_scale=1.0, seed=0):
    rng = np.random.default_rng(seed)
    x = np.asarray(initial, dtype=float).copy()
    trajectory = [x.copy()]
    for _ in range(steps):
        drift = -0.5 * eta * grad_energy(x)
        noise = noise_scale * np.sqrt(eta) * rng.normal(size=x.shape)
        x = x + drift + noise
        trajectory.append(x.copy())
    return np.asarray(trajectory)

x0 = 1.50
eta = 0.04                      # matches slide 'One Langevin step, computed by hand'
gradient = float(grad_energy(np.array([x0]))[0])   # 4·1.5·(1.5²−1) = 7.5
realised_noise = np.sqrt(eta) * 0.25   # ε = 0.25 → 0.05
after_drift = x0 - eta / 2 * gradient
after_noise = after_drift + realised_noise
print(f"current {x0:.2f} → drift {after_drift:.2f} → noise {after_noise:.2f}")

## 3. Chains and coverage

Predict whether each noise setting will cross the central barrier.

In [ ]:
settings = [(0.0, "no noise"), (0.45, "low noise"), (1.0, "Langevin noise")]
initial = np.linspace(-1.8, 1.8, 24)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), gridspec_kw={"height_ratios": [2, 1]})
for column, (scale, label) in enumerate(settings):
    trajectories = langevin(initial, steps=600, eta=0.02, noise_scale=scale, seed=20 + column)
    for chain in range(len(initial)):
        axes[0, column].plot(trajectories[:, chain], alpha=0.35, lw=0.8)
    axes[0, column].set_title(label)
    axes[0, column].set_xlabel("step"); axes[0, column].set_ylabel("state x")
    axes[0, column].axhline(-1, color="tab:green", ls="--", lw=1)
    axes[0, column].axhline(1, color="tab:green", ls="--", lw=1)
    samples = trajectories[200:].reshape(-1)
    axes[1, column].hist(samples, bins=45, density=True, color="tab:blue", alpha=0.75)
    axes[1, column].set_xlabel("visited state"); axes[1, column].set_ylabel("density")
fig.suptitle("Noise changes exploration and empirical mode coverage")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "04_ebm_langevin.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. What EBM training is trying to do

The model is pθ(x) = exp(−Eθ(x)) / Zθ. For the negative log-likelihood,

∇θ L = E_data[∇θEθ(x)] − E_model[∇θEθ(x)].

Gradient descent therefore has two opposing effects:

- **positive phase:** lower Eθ on observed data;
- **negative phase:** raise Eθ on samples currently preferred by the model.

The exact negative phase is an expectation under pθ and therefore requires sampling. Contrastive divergence replaces it with short MCMC chains. In the code below, `mean(E(data)) − mean(E(negative particles))` is the direct approximation of these two terms.

In [ ]:
states = np.array([-1.0, 0.0, 1.0])
energies_before = np.array([0.8, 0.2, 0.9])
prob_before = np.exp(-energies_before); prob_before /= prob_before.sum()
energies_after = np.array([0.5, 0.8, 0.6])  # lower data energies at ±1, raise model-favoured x=0
prob_after = np.exp(-energies_after); prob_after /= prob_after.sum()
print("state             ", states)
print("probability before", prob_before.round(3), "← model wrongly prefers x=0")
print("probability after ", prob_after.round(3), "← mass moves toward the data modes")
print("The partition function changes automatically when any energy changes.")

## 5. Train a one-dimensional neural EBM

Data come from two Gaussian modes. Persistent negative particles approximate samples from the current model. Each update lowers the data energy and raises the negative-particle energy.

In [ ]:
import os
import torch
from torch import nn
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
QUICK_MODE = os.getenv("LECTURE2_QUICK", "0") == "1"
artifact_suffix = "_smoke" if QUICK_MODE else ""

class EnergyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def draw_data(count):
    modes = torch.randint(0, 2, (count, 1), device=device) * 2 - 1
    return 1.5 * modes.float() + 0.18 * torch.randn(count, 1, device=device)

def update_negative(model, particles, steps=12, step_size=0.025, noise=1.0):
    x = particles.detach()
    for _ in range(steps):
        x.requires_grad_(True)
        gradient = torch.autograd.grad(model(x).sum(), x)[0]
        x = (x - 0.5 * step_size * gradient + noise * step_size**0.5 * torch.randn_like(x)).detach()
        x.clamp_(-3.0, 3.0)
    return x

energy_model = EnergyNet().to(device)
optimizer = torch.optim.Adam(energy_model.parameters(), lr=2e-3)
negative_particles = torch.empty(256, 1, device=device).uniform_(-3, 3)
training_steps = 800 if not QUICK_MODE else 140
grid_t = torch.linspace(-3, 3, 400, device=device)[:, None]
snapshots, snapshot_steps, losses = [], [], []

for step in range(training_steps + 1):
    data_batch = draw_data(256)
    negative_particles = update_negative(energy_model, negative_particles)
    positive_energy = energy_model(data_batch)
    negative_energy = energy_model(negative_particles)
    contrastive_loss = positive_energy.mean() - negative_energy.mean()
    regularizer = 1e-3 * (positive_energy.square().mean() + negative_energy.square().mean())
    loss = contrastive_loss + regularizer
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
    if step % max(training_steps // 12, 1) == 0 or step == training_steps:
        with torch.no_grad():
            curve = energy_model(grid_t).cpu().numpy()
        snapshots.append((curve, negative_particles.detach().cpu().numpy().copy()))
        snapshot_steps.append(step)

print("device:", device)
print(f"contrastive objective: {losses[0]:.3f} → {losses[-1]:.3f}")

## 6. Watch the energy landscape and negative particles co-evolve

The absolute energy level is arbitrary. What matters is the relative landscape: low near the two data modes, higher between and outside them.

In [ ]:
grid_np = grid_t.cpu().numpy().ravel()
data_for_plot = draw_data(1000).cpu().numpy().ravel()
fig, ax = plt.subplots(figsize=(7.5, 4.2))
curve_line, = ax.plot([], [], lw=2.5, color="tab:blue", label="learned energy")
particle_marks = ax.scatter([], [], marker="|", s=130, color="tab:orange", alpha=0.65, label="negative particles")
ax.hist(data_for_plot, bins=55, density=True, alpha=0.18, color="tab:green", label="data density")
ax.set_xlim(-3, 3); ax.set_ylim(-1.5, 5.0); ax.set_xlabel("x"); ax.set_ylabel("energy / scaled density")
ax.legend(loc="upper right")
def update_ebm(frame_index):
    curve, particles = snapshots[frame_index]
    curve = curve - curve.min()
    curve_line.set_data(grid_np, curve)
    particle_marks.set_offsets(np.c_[particles.ravel(), np.full(len(particles), 0.12)])
    ax.set_title(f"EBM training step {snapshot_steps[frame_index]}")
    return curve_line, particle_marks
ebm_animation = FuncAnimation(fig, update_ebm, frames=len(snapshots), interval=650, blit=False)
ebm_gif = ROOT / "fallbacks" / f"04_ebm_training{artifact_suffix}.gif"
ebm_animation.save(ebm_gif, writer=PillowWriter(fps=2))
plt.close(fig)
display(HTML(ebm_animation.to_jshtml()))
print("saved", ebm_gif)

final_curve, final_particles = snapshots[-1]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(grid_np, final_curve - final_curve.min(), lw=2.5, label="learned Eθ(x)")
ax.hist(data_for_plot, bins=55, density=True, alpha=0.25, label="data")
ax.scatter(final_particles.ravel(), np.full(len(final_particles), 0.08), marker="|", s=90, alpha=0.45, label="model particles")
ax.set_xlabel("x"); ax.set_ylabel("relative energy / density"); ax.legend()
ax.set_title("Training lowers energy near data and moves model particles toward both modes")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / f"04_ebm_trained{artifact_suffix}.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Where the approximation can fail

If the negative chains do not mix, training raises energy only where those chains happen to visit. The model may keep unseen low-energy regions. This is why sampling quality and the learning objective cannot be assessed separately for an EBM.

## Interpretation check and optional extension

Separate the energy model from the sampler. Explain how poor mixing can corrupt both generated samples and the negative phase used for learning.

**Optional extension:** start all persistent particles in one well and test which step-size and noise combinations recover both modes.

Record the changed parameter, the fixed controls, the observed evidence, and one limitation.